# Colab GPU Training Notebook

Trains the multi-head 3D SE-ResNet50 model on GPU, with GradNorm balancing
the 10 per-head regression losses. Each head predicts a confidence score in
[0, 1] (sigmoid output), trained with MSE against the manifest's `_score`
columns.

Upload the LungInsight repository content to `/content/LungInsight` before
running (it must contain `cir_multihead_pipeline.py` and `se_resnet3d.py`).

In [ ]:
%pip install -q torch torchvision numpy pandas scikit-learn pylidc gradnorm-pytorch grad-cam

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
DRIVE_DIR = '/content/drive/MyDrive/lunginsight'
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(ROOT_DIR):
    raise RuntimeError(
        'Please upload repository content (including cir_multihead_pipeline.py '
        'and se_resnet3d.py) to /content/LungInsight before running this notebook.'
    )
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn.functional as F

from gradnorm_pytorch import GradNormLossWeighter

from cir_multihead_pipeline import create_multihead_model, FEATURE_NAMES, LIDCPatchDataset

## Config and data loaders

In [ ]:
TRAIN_CSV = '/content/drive/MyDrive/lunginsight/cpu_split/train_split.csv'
VAL_CSV = '/content/drive/MyDrive/lunginsight/cpu_split/val_split.csv'
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
GRADNORM_LR = 1e-4
RESTORING_FORCE_ALPHA = 0.0  # 0.0 = perfectly balanced losses; paper goes up to ~3
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Dataset tensors are moved to the right device per-sample below; keep the
# Dataset itself on CPU so DataLoader workers (num_workers > 0) can pickle it.
train_dataset = LIDCPatchDataset(TRAIN_CSV, device='cpu')
val_dataset = LIDCPatchDataset(VAL_CSV, device='cpu')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

## Model, optimizer, and GradNorm loss weighter

`GradNormLossWeighter` needs a reference parameter tensor (typically the
last shared-trunk layer) to measure each head's gradient norm against. We
use the final backbone layer (`layer4`'s last block's `conv3` weight) since
it's the last point all 10 heads share before branching.

Note: `model.layer4[-1].conv3.weight` assumes layer4 is an `nn.Sequential`
of `SEBottleneck3D` blocks, matching se_resnet3d.py's current
implementation. If the backbone implementation changes, update this line
to point at whatever the new final shared-trunk weight tensor is.

In [ ]:
model = create_multihead_model(head_names=FEATURE_NAMES, device=device)
optimizer = Adam(model.parameters(), lr=LR)

gradnorm_parameter = model.layer4[-1].conv3.weight

loss_weighter = GradNormLossWeighter(
    num_losses=len(FEATURE_NAMES),
    learning_rate=GRADNORM_LR,
    restoring_force_alpha=RESTORING_FORCE_ALPHA,
    grad_norm_parameters=gradnorm_parameter,
)

## Training and validation loops

Per-head loss is MSE between the sigmoid confidence score and the
manifest's normalized `_score` target. `loss_weighter.backward(losses)`
replaces the usual `loss.backward()` call -- GradNorm computes and applies
the backward pass internally using the per-task loss list.

In [ ]:
def train_one_epoch(loader, model, loss_weighter, optimizer, device):
    model.train()
    running = {feat: 0.0 for feat in FEATURE_NAMES}
    n_batches = 0

    for patches, targets in loader:
        patches = patches.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}

        optimizer.zero_grad()
        outputs = model(patches)

        # Order must be stable and match num_losses=len(FEATURE_NAMES).
        losses = [F.mse_loss(outputs[feat], targets[feat]) for feat in FEATURE_NAMES]

        loss_weighter.backward(losses)
        optimizer.step()

        for feat, l in zip(FEATURE_NAMES, losses):
            running[feat] += l.item()
        n_batches += 1

    return {feat: running[feat] / max(n_batches, 1) for feat in FEATURE_NAMES}


def validate_one_epoch(loader, model, device):
    model.eval()
    running = {feat: 0.0 for feat in FEATURE_NAMES}
    n_batches = 0

    with torch.no_grad():
        for patches, targets in loader:
            patches = patches.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}
            outputs = model(patches)
            for feat in FEATURE_NAMES:
                running[feat] += F.mse_loss(outputs[feat], targets[feat]).item()
            n_batches += 1

    return {feat: running[feat] / max(n_batches, 1) for feat in FEATURE_NAMES}

## Run training

In [ ]:
best_val_loss = float('inf')
best_model_path = os.path.join(DRIVE_DIR, 'best_model_gpu.pth')

for epoch in range(1, EPOCHS + 1):
    print(f'=== Epoch {epoch}/{EPOCHS} ===')
    train_loss = train_one_epoch(train_loader, model, loss_weighter, optimizer, device)
    val_loss = validate_one_epoch(val_loader, model, device)

    print('Training losses:', {k: round(v, 4) for k, v in train_loss.items()})
    print('Validation losses:', {k: round(v, 4) for k, v in val_loss.items()})

    avg_val = sum(val_loss.values()) / len(val_loss)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), best_model_path)
        print(f'Saved best model to {best_model_path} (avg val loss {best_val_loss:.4f})')